# Day 09. Exercise 00
# Regularization

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/dayofweek.csv')

In [3]:
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [5]:
model = LogisticRegression(
    random_state=21,
    fit_intercept=False,
    max_iter=1000
)

In [6]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

    model.fit(X_tr, y_tr)

    train_pred = model.predict(X_tr)
    valid_pred = model.predict(X_val)

    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)

    train_scores.append(train_acc)
    valid_scores.append(valid_acc)

    print(f"train - {train_acc:.5f} | valid - {valid_acc:.5f}")

train - 0.64056 | valid - 0.65926
train - 0.63561 | valid - 0.62222
train - 0.64468 | valid - 0.60000
train - 0.64056 | valid - 0.64444
train - 0.65375 | valid - 0.60741
train - 0.62902 | valid - 0.60000
train - 0.66117 | valid - 0.60000
train - 0.63726 | valid - 0.54074
train - 0.63756 | valid - 0.66418
train - 0.64745 | valid - 0.61194


In [7]:
%%time

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

    model.fit(X_tr, y_tr)

    train_pred = model.predict(X_tr)
    valid_pred = model.predict(X_val)

    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)

    train_scores.append(train_acc)
    valid_scores.append(valid_acc)

    print(f"train - {train_acc:.5f} | valid - {valid_acc:.5f}")

train - 0.64056 | valid - 0.65926
train - 0.63561 | valid - 0.62222
train - 0.64468 | valid - 0.60000
train - 0.64056 | valid - 0.64444
train - 0.65375 | valid - 0.60741
train - 0.62902 | valid - 0.60000
train - 0.66117 | valid - 0.60000
train - 0.63726 | valid - 0.54074
train - 0.63756 | valid - 0.66418
train - 0.64745 | valid - 0.61194
CPU times: total: 359 ms
Wall time: 420 ms


In [8]:
print(f"\nAverage accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")


Average accuracy on crossval is 0.61502
Std is 0.03399


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [9]:
model = LogisticRegression(
    penalty='none',
    random_state=21,
    fit_intercept=False,
    max_iter=1000
)

In [10]:
model = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    random_state=21,
    fit_intercept=False,
    max_iter=1000
)

In [11]:
model = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    random_state=21,
    fit_intercept=False,
    max_iter=1000
)

## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [12]:
model = SVC(
    kernel='linear',
    probability=True,
    random_state=21
)

In [13]:
%%time

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

    model.fit(X_tr, y_tr)

    train_acc = accuracy_score(y_tr, model.predict(X_tr))
    valid_acc = accuracy_score(y_val, model.predict(X_val))

    train_scores.append(train_acc)
    valid_scores.append(valid_acc)

    print(f"train - {train_acc:.5f} | valid - {valid_acc:.5f}")

print(f"\nAverage accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

train - 0.70651 | valid - 0.68148
train - 0.68920 | valid - 0.64444
train - 0.69744 | valid - 0.66667
train - 0.68920 | valid - 0.65926
train - 0.69497 | valid - 0.63704
train - 0.68673 | valid - 0.68148
train - 0.69827 | valid - 0.61481
train - 0.70486 | valid - 0.57778
train - 0.68863 | valid - 0.72388
train - 0.71005 | valid - 0.64179

Average accuracy on crossval is 0.65286
Std is 0.03800
CPU times: total: 4.11 s
Wall time: 5.9 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [14]:
C_values = [0.01, 0.1, 1, 10, 100]

In [15]:
for C in C_values:
    print(f"\n=== C = {C} ===")
    
    model = SVC(
        kernel='linear',
        probability=True,
        random_state=21,
        C=C
    )
    
    valid_scores = []
    
    for train_idx, valid_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model.fit(X_tr, y_tr)
        valid_acc = accuracy_score(y_val, model.predict(X_val))
        valid_scores.append(valid_acc)

    print(f"Avg accuracy: {np.mean(valid_scores):.5f}")
    print(f"Std: {np.std(valid_scores):.5f}")


=== C = 0.01 ===
Avg accuracy: 0.38125
Std: 0.02293

=== C = 0.1 ===
Avg accuracy: 0.55787
Std: 0.02522

=== C = 1 ===
Avg accuracy: 0.65286
Std: 0.03800

=== C = 10 ===
Avg accuracy: 0.71814
Std: 0.04026

=== C = 100 ===
Avg accuracy: 0.75001
Std: 0.02849


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [16]:
model = DecisionTreeClassifier(
    max_depth=10,
    random_state=21
)

In [17]:
%%time

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

    model.fit(X_tr, y_tr)

    train_acc = accuracy_score(y_tr, model.predict(X_tr))
    valid_acc = accuracy_score(y_val, model.predict(X_val))

    train_scores.append(train_acc)
    valid_scores.append(valid_acc)

    print(f"train - {train_acc:.5f} | valid - {valid_acc:.5f}")

print(f"\nAverage accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

train - 0.80874 | valid - 0.77037
train - 0.79802 | valid - 0.70370
train - 0.81286 | valid - 0.72593
train - 0.80049 | valid - 0.74815
train - 0.80956 | valid - 0.68889
train - 0.78978 | valid - 0.74074
train - 0.80627 | valid - 0.60741
train - 0.82688 | valid - 0.71111
train - 0.78995 | valid - 0.79104
train - 0.80313 | valid - 0.70896

Average accuracy on crossval is 0.71963
Std is 0.04791
CPU times: total: 188 ms
Wall time: 217 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [18]:
depths = [2, 3, 5, 7, 10, 15, 20]

In [19]:
for depth in depths:
    print(f"\n=== max_depth = {depth} ===")
    
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=21
    )
    
    valid_scores = []
    
    for train_idx, valid_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model.fit(X_tr, y_tr)
        valid_acc = accuracy_score(y_val, model.predict(X_val))
        valid_scores.append(valid_acc)

    print(f"Avg accuracy: {np.mean(valid_scores):.5f}")
    print(f"Std: {np.std(valid_scores):.5f}")


=== max_depth = 2 ===
Avg accuracy: 0.43467
Std: 0.02310

=== max_depth = 3 ===
Avg accuracy: 0.47031
Std: 0.01864

=== max_depth = 5 ===
Avg accuracy: 0.54448
Std: 0.04014

=== max_depth = 7 ===
Avg accuracy: 0.65284
Std: 0.04153

=== max_depth = 10 ===
Avg accuracy: 0.71963
Std: 0.04791

=== max_depth = 15 ===
Avg accuracy: 0.85682
Std: 0.02780

=== max_depth = 20 ===
Avg accuracy: 0.88798
Std: 0.01862


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [20]:
model = RandomForestClassifier(
    n_estimators=50,
    max_depth=14,
    random_state=21
)

In [21]:
%%time

train_scores = []
valid_scores = []

for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

    model.fit(X_tr, y_tr)

    train_acc = accuracy_score(y_tr, model.predict(X_tr))
    valid_acc = accuracy_score(y_val, model.predict(X_val))

    train_scores.append(train_acc)
    valid_scores.append(valid_acc)

    print(f"train - {train_acc:.5f} | valid - {valid_acc:.5f}")

print(f"\nAverage accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

train - 0.97939 | valid - 0.85185
train - 0.96620 | valid - 0.85926
train - 0.96208 | valid - 0.91852
train - 0.97115 | valid - 0.91852
train - 0.97197 | valid - 0.88148
train - 0.96538 | valid - 0.86667
train - 0.96455 | valid - 0.88889
train - 0.96867 | valid - 0.87407
train - 0.96458 | valid - 0.93284
train - 0.96787 | valid - 0.86567

Average accuracy on crossval is 0.88578
Std is 0.02673
CPU times: total: 1.64 s
Wall time: 2.26 s


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [22]:
depths = [5, 10, 14, 20]
estimators = [10, 50, 100]

In [23]:
for depth in depths:
    for n in estimators:
        print(f"\n=== depth={depth}, n_estimators={n} ===")
        
        model = RandomForestClassifier(
            max_depth=depth,
            n_estimators=n,
            random_state=21
        )
        
        valid_scores = []
        
        for train_idx, valid_idx in skf.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

            model.fit(X_tr, y_tr)
            valid_acc = accuracy_score(y_val, model.predict(X_val))
            valid_scores.append(valid_acc)

        print(f"Avg accuracy: {np.mean(valid_scores):.5f}")
        print(f"Std: {np.std(valid_scores):.5f}")


=== depth=5, n_estimators=10 ===
Avg accuracy: 0.58752
Std: 0.02557

=== depth=5, n_estimators=50 ===
Avg accuracy: 0.57788
Std: 0.04236

=== depth=5, n_estimators=100 ===
Avg accuracy: 0.56971
Std: 0.03398

=== depth=10, n_estimators=10 ===
Avg accuracy: 0.77819
Std: 0.03673

=== depth=10, n_estimators=50 ===
Avg accuracy: 0.79674
Std: 0.04656

=== depth=10, n_estimators=100 ===
Avg accuracy: 0.80713
Std: 0.03365

=== depth=14, n_estimators=10 ===
Avg accuracy: 0.86650
Std: 0.02400

=== depth=14, n_estimators=50 ===
Avg accuracy: 0.88578
Std: 0.02673

=== depth=14, n_estimators=100 ===
Avg accuracy: 0.89022
Std: 0.02668

=== depth=20, n_estimators=10 ===
Avg accuracy: 0.89246
Std: 0.02651

=== depth=20, n_estimators=50 ===
Avg accuracy: 0.91320
Std: 0.03386

=== depth=20, n_estimators=100 ===
Avg accuracy: 0.91542
Std: 0.03152


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [24]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=14,
    random_state=21
)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,14
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [25]:
#final accuracy
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Test accuracy: {acc:.5f}")

Test accuracy: 0.89645


In [26]:
#error for predicting each day
df_res = pd.DataFrame({
    'true': y_test,
    'pred': y_pred
})

errors = df_res[df_res['true'] != df_res['pred']]

error_percent = (
    errors['true'].value_counts() /
    df_res['true'].value_counts()
) * 100

print(error_percent.sort_values(ascending=False))

true
0    25.925926
4    19.047619
1    14.545455
6    11.267606
5     7.407407
2     6.666667
3     2.500000
Name: count, dtype: float64


In [27]:
df_pred = pd.DataFrame({
    'prediction': y_pred
})

In [28]:
#saving model
joblib.dump(model, '../data/best_model.pkl')

['../data/best_model.pkl']